# Documentos de carta enriquecidos com o dicionario de keywords (Etapa 1.1)

*(Feito com ClaudeAI)*

Junta o texto de cada carta (do bulk data "Oracle Cards" da Scryfall) com
as definicoes do `keyword_dictionary.json`, gerando as representacoes de
texto usadas na etapa de embedding, mais os dados estruturados relevantes
pro jogo (nome, custo, cor, poder/resistencia/lealdade quando aplicavel).

## Configuracao

In [3]:
import json
import os
import re

CARDS_PATH = "Dados/Raw/oracle-cards.jsonl"
KEYWORD_DICT_PATH = "Dados/Etapa1_card_expanded/keyword_dictionary.json"
ABILITY_WORDS_PATH = "Dados/Etapa1_card_expanded/keyword_dictionary_ability_words.json"
OUTPUT_PATH = "Dados/Etapa1_card_expanded/card_documents.jsonl"

# layouts que NAO sao cartas de baralho de verdade -- tokens, emblemas,
# cartas so de arte (Secret Lair "art_series"), e objetos de formatos
# alternativos (Vanguard/Archenemy/Planechase) nao fazem sentido num
# corpus de busca pra montar deck.
EXCLUDED_LAYOUTS = {
    "token", "double_faced_token", "emblem", "art_series",
    "vanguard", "scheme", "planar", "front_card",
}

# Un-sets (bordas prateadas -- Unglued/Unhinged/Unstable/Unfinity etc.)
# excluidos por set_type, nao por layout: cartas como "Everythingamajig"
# usam layout "normal" mas tem o MESMO nome com textos diferentes de
# proposito (gimmick de design), o que so o filtro por set_type pega
# por completo.
EXCLUDED_SET_TYPES = {"funny"}

# otags (etiquetas funcionais do Scryfall Tagger) usadas pra enriquecer o
# texto de embedding. Buscadas via API (`otag:<tag>`) -- nao existem no
# bulk data.
OTAGS = ["ramp", "card-advantage", "removal"]
OTAGS_CACHE_DIR = "Dados/Etapa1_card_expanded/otag_cache"   # uma resposta por tag
OTAGS_PATH = "Dados/Etapa1_card_expanded/card_otags.json"    # carta -> [tags]

## Extrair o texto da carta (e preservar as faces)

Cartas normais tem `oracle_text` no nivel principal. Cartas de dupla face
(layout `transform`, `modal_dfc`, etc) **nao tem** -- o texto fica dentro de
`card_faces`. A versao anterior so concatenava as faces com `" // "` e
perdia a estrutura; agora `get_card_faces()` devolve as faces como lista
estruturada (nome/type_line/mana_cost/oracle_text de cada uma), e
`get_card_text()` continua gerando a versao concatenada pra quem preferir
um unico texto.

In [4]:
def get_card_faces(card: dict):
    faces = card.get("card_faces")
    if not faces:
        return None
    return [
        {
            "name": f.get("name"),
            "type_line": f.get("type_line"),
            "mana_cost": f.get("mana_cost"),
            "oracle_text": f.get("oracle_text"),
            "power": f.get("power"),
            "toughness": f.get("toughness"),
        }
        for f in faces
    ]


def get_card_text(card: dict) -> str:
    if card.get("oracle_text"):
        return card["oracle_text"]
    faces = card.get("card_faces")
    if faces:
        parts = [f.get("oracle_text") or "vanilla" for f in faces]
        return " // ".join(parts)
    return "vanilla"

## Decodificar simbolos de mana e ativacao

`oracle_text` e `mana_cost` usam simbolos tipo `{T}` (virar), `{G}` (mana
verde), `{2/W}` (hibrido monocolorido) -- um modelo de embedding generico
nao tem por que saber o que esses simbolos significam, sao notacao
especifica do jogo. A regra 107 da propria Comprehensive Rules define
todos eles (107.4 pros simbolos de mana, 107.5/107.6 pro simbolo de virar/
desvirar), entao construimos o glossario de simbolos a partir dai, em vez
de eu confiar so na minha memoria sobre o assunto.

**Importante: isso so afeta o campo `embedding_text`.** O campo
`oracle_text` continua com os simbolos originais intactos -- a decodificacao
e' so uma transformacao pra ajudar o embedding, nao uma edicao dos dados
de origem (mesmo principio de manter raw + versao processada que usamos
no dicionario de keywords).

**Simbolos repetidos em sequencia sao agrupados:** `{C}{C}` vira `two
colorless mana` (e nao `colorless mana colorless mana`), `{G}{G}` vira
`two green mana`, e assim por diante. Isso vale so pra simbolos de mana
"puros" (cores, incolor, neve, X e hibridos de duas cores). Numeros
(`{1}{1}`), `{T}` e os simbolos com custo alternativo (`{2/W}`, `{G/P}`)
continuam um a um, porque "two 2 generic mana or one white mana" nao
leria bem. Simbolos diferentes lado a lado (`{5}{G}{G}`) seguem
separados por espaco: `5 generic mana two green mana`.

In [5]:
from itertools import groupby

COLOR_NAMES = {"W": "white", "U": "blue", "B": "black", "R": "red", "G": "green"}
HYBRID_PAIRS = [("W","U"),("W","B"),("U","B"),("U","R"),("B","R"),
                ("B","G"),("R","G"),("R","W"),("G","W"),("G","U")]

SYMBOL_GLOSSARY = {}
for c, name in COLOR_NAMES.items():
    SYMBOL_GLOSSARY[c] = f"{name} mana"
SYMBOL_GLOSSARY["C"] = "colorless mana"
SYMBOL_GLOSSARY["S"] = "snow mana"
SYMBOL_GLOSSARY["X"] = "X generic mana"
SYMBOL_GLOSSARY["T"] = "tap this permanent"
SYMBOL_GLOSSARY["Q"] = "untap this permanent"
for n in range(0, 21):
    SYMBOL_GLOSSARY[str(n)] = f"{n} generic mana"
for a, b in HYBRID_PAIRS:
    SYMBOL_GLOSSARY[f"{a}/{b}"] = f"{COLOR_NAMES[a]} or {COLOR_NAMES[b]} mana"
for c, name in COLOR_NAMES.items():
    SYMBOL_GLOSSARY[f"2/{c}"] = f"two generic mana or one {name} mana"
    SYMBOL_GLOSSARY[f"C/{c}"] = f"one colorless mana or one {name} mana"
    SYMBOL_GLOSSARY[f"{c}/P"] = f"one {name} mana or 2 life"
for a, b in HYBRID_PAIRS:
    SYMBOL_GLOSSARY[f"{a}/{b}/P"] = f"{COLOR_NAMES[a]} mana, {COLOR_NAMES[b]} mana, or 2 life"

NUMBER_WORDS = {2: "two", 3: "three", 4: "four", 5: "five", 6: "six", 7: "seven",
                8: "eight", 9: "nine", 10: "ten", 11: "eleven", 12: "twelve",
                13: "thirteen", 14: "fourteen", 15: "fifteen", 16: "sixteen"}

# so esses simbolos sao agrupados quando se repetem em sequencia; numeros,
# {T}/{Q} e simbolos com custo alternativo ({2/W}, {G/P}) ficam um a um.
AGGREGATABLE = (set(COLOR_NAMES) | {"C", "S", "X"}
                | {f"{a}/{b}" for a, b in HYBRID_PAIRS})


def _symbol_parts(symbols: list) -> list:
    """Descricao de cada simbolo de uma sequencia colada, ex.: ['5', 'G', 'G']
    -> ['5 generic mana', 'two green mana']."""
    parts = []
    for sym, group in groupby(symbols):
        n = len(list(group))
        desc = SYMBOL_GLOSSARY.get(sym)
        if desc is None:                        # simbolo desconhecido: mantem como esta
            parts.extend([f"{{{sym}}}"] * n)
        elif n > 1 and sym in AGGREGATABLE:
            parts.append(f"{NUMBER_WORDS.get(n, n)} {desc}")
        else:
            parts.extend([desc] * n)
    return parts


def _expand_run(match) -> str:
    """Expande uma sequencia colada de simbolos, ex.: {5}{G}{G}."""
    symbols = re.findall(r"\{([^{}]+)\}", match.group(0))
    return " " + " ".join(_symbol_parts(symbols)) + " "


def expand_symbols(text: str) -> str:
    if not text:
        return text
    text = re.sub(r"(?:\{[^{}]+\})+", _expand_run, text)
    return re.sub(r"\s{2,}", " ", text).strip()


print(f"{len(SYMBOL_GLOSSARY)} simbolos mapeados a partir da regra 107")
print(expand_symbols("{T}: Add {G}. Trample {2}{R}."))
print(expand_symbols("{T}: Add {C}{C}. {5}{G}{G}: Transform this creature."))


66 simbolos mapeados a partir da regra 107
tap this permanent : Add green mana . Trample 2 generic mana red mana .
tap this permanent : Add two colorless mana . 5 generic mana two green mana : Transform this creature.


## Reescrever habilidades ativadas ("custo: efeito")

Habilidades ativadas vem no formato `custo: efeito` (`{T}: Add {G}.`).
Pra um modelo de linguagem, essa notacao com dois-pontos e' mais opaca do
que uma frase, entao reescrevemos como **"faca X para fazer Y"**:

- `{T}: Add {G}.` -> `Tap this permanent to add green mana.`
- `{5}{G}{G}: Transform this creature.` -> `Pay 5 generic mana and two green mana to transform this creature.`
- `{2}{R}, {T}, Discard a card: Draw a card.` -> `Pay 2 generic mana and red mana, tap this permanent, and discard a card to draw a card.`
- `+1: You gain 2 life.` -> `Put 1 loyalty counter on this planeswalker so that you gain 2 life.`

Decisoes:
- **`to` x `so that`:** se o efeito comeca com um verbo no imperativo
  (`Add`, `Draw`, `Destroy`...), usamos `to`. Se comeca com uma oracao
  completa (`Target creature gets +1/+1`, `Shivan Dragon gets +1/+0`),
  `to` ficaria agramatical, entao usamos `so that`.
- **Custo de mana** ganha o verbo `pay`; `{T}` ja vira "tap this permanent".
  Custos de lealdade seguem a regra 606.4 (colocar/remover marcadores).
- **So mexe no que parece habilidade ativada:** se qualquer parte do custo
  nao for simbolo, lealdade ou um verbo de custo conhecido (`Discard`,
  `Sacrifice`, `Pay`...), a linha fica como esta. Um ":" qualquer no meio
  do texto nao e' reescrito.
- Habilidades dentro de aspas (`have "{T}: Add {G}."`) e de reminder text
  entre parenteses (`({T}: Add {G}.)`, o texto dos terrenos basicos)
  tambem sao reescritas, pra ficarem iguais as demais.
- Cada linha, e cada face (separadas por ` // `), e' tratada isoladamente.
- Roda ANTES de `expand_symbols`: precisa dos simbolos originais pra
  diferenciar custo de mana de `{T}`.


In [6]:
# Habilidades ativadas "custo: efeito" -> "Faca o custo to fazer o efeito".
# Trabalha no texto ORIGINAL (com simbolos), porque precisa distinguir custo
# de mana ({2}{R}) de {T} e de custos de lealdade (+1, -2).

# Efeito que comeca com verbo no imperativo ("Add {G}", "Draw a card") combina
# com "to" -> "Tap this permanent to add green mana."
IMPERATIVE_VERBS = {
    "activate", "add", "adapt", "amass", "attach", "bolster", "cast", "change",
    "choose", "cloak", "connive", "copy", "counter", "create", "destroy",
    "detain", "discard", "discover", "distribute", "double", "draw", "exchange",
    "exile", "explore", "fateseal", "fight", "flip", "forage", "gain", "goad",
    "incubate", "investigate", "learn", "level", "look", "manifest", "mill",
    "monstrosity", "move", "play", "populate", "prevent", "proliferate", "put",
    "reduce", "regenerate", "remove", "return", "reveal", "roll", "sacrifice",
    "scry", "search", "set", "shuffle", "surveil", "switch", "tap", "transform",
    "triple", "turn", "unattach", "untap", "venture", "vote",
}
# Efeito que comeca com uma oracao completa ("Target creature gets +1/+1",
# "Shivan Dragon gets +1/+0") usa "so that" -> "... so that target creature gets +1/+1".
# Esses inicios de frase sao minusculizados; nomes de carta ficam como estao.
CLAUSE_STARTERS = {
    "target", "you", "each", "all", "this", "that", "another", "up", "it", "its",
    "the", "a", "an", "if", "until", "enchanted", "equipped", "other", "your",
    "creatures", "players", "opponents", "any",
}
# Verbos que comecam um custo que nao e' simbolo ("Discard a card", "Pay 3 life")
COST_VERBS = {
    "Discard", "Sacrifice", "Pay", "Exile", "Remove", "Return", "Tap", "Untap",
    "Reveal", "Put", "Exert", "Mill", "Forage", "Collect",
}
LOYALTY_RE = re.compile(r"^([+\u2212\-])?(\d+|X)$")   # +1, -2 (U+2212), 0, -X


def _lower_first(s: str) -> str:
    return s[:1].lower() + s[1:]


def _upper_first(s: str) -> str:
    return s[:1].upper() + s[1:]


def _join_and(items: list) -> str:
    if len(items) <= 2:
        return " and ".join(items)
    return ", ".join(items[:-1]) + ", and " + items[-1]


def _cost_piece(piece: str):
    """Converte uma parte do custo (entre virgulas) em texto simples;
    devolve None se nao parece um custo."""
    piece = piece.strip()
    if re.fullmatch(r"(?:\{[^{}]+\})+", piece):
        symbols = re.findall(r"\{([^{}]+)\}", piece)
        if not all(s in SYMBOL_GLOSSARY for s in symbols):
            return None
        parts = _symbol_parts(symbols)
        if any(s in ("T", "Q") for s in symbols):
            return " ".join(parts)                      # {T} -> "tap this permanent"
        return "pay " + _join_and(parts)                # {2}{R} -> "pay 2 generic mana and red mana"
    m = LOYALTY_RE.match(piece)
    if m:                                               # custo de lealdade (CR 606.4)
        sign, amount = m.groups()
        if sign is None:
            return "pay no loyalty cost" if amount == "0" else None
        noun = f"{amount} loyalty counter" + ("" if amount == "1" else "s")
        return f"put {noun} on this planeswalker" if sign == "+" else f"remove {noun} from this planeswalker"
    if piece.split(" ", 1)[0] in COST_VERBS:
        return _lower_first(expand_symbols(piece))
    return None


def _parse_cost(cost: str):
    pieces = [_cost_piece(p) for p in cost.strip().split(", ")] if cost.strip() else []
    return None if not pieces or any(p is None for p in pieces) else pieces


def _transform_segment(seg: str) -> str:
    # habilidades dentro de aspas ("...have '{T}: Add {G}.'") e de reminder
    # text entre parenteses ("({T}: Add {G}.)") tambem sao reescritas
    seg = re.sub(r'"([^"]*)"', lambda m: '"' + _transform_segment(m.group(1)) + '"', seg)
    seg = re.sub(r"\(([^()]*)\)", lambda m: "(" + _transform_segment(m.group(1)) + ")", seg)

    depth, in_quote, colon = 0, False, None             # primeiro ":" fora de aspas/parenteses
    for i, ch in enumerate(seg):
        if ch == '"':
            in_quote = not in_quote
        elif in_quote:
            continue
        elif ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        elif ch == ":" and depth == 0:
            colon = i
            break
    if colon is None:
        return seg

    cost, effect = seg[:colon], seg[colon + 1:].strip()
    phrases = _parse_cost(cost)
    if not phrases or not effect:                       # nao parece habilidade ativada: nao mexe
        return seg

    m = re.match(r"[A-Za-z']+", effect)
    word = m.group(0).lower() if m else ""
    if word in IMPERATIVE_VERBS:
        connector, effect = " to ", _lower_first(effect)
    else:
        connector = " so that "
        if word in CLAUSE_STARTERS:
            effect = _lower_first(effect)
    return _upper_first(_join_and(phrases)) + connector + effect


def restructure_activated_abilities(text: str) -> str:
    """'{T}: Add {G}.' -> 'Tap this permanent to add {G}.' (simbolos do efeito
    continuam como estao; expand_symbols cuida deles depois)."""
    if not text:
        return text
    # cada linha e cada face (separadas por " // ") e' tratada isoladamente
    parts = re.split(r"(\n| // )", text)
    return "".join(p if i % 2 else _transform_segment(p) for i, p in enumerate(parts))


print(expand_symbols(restructure_activated_abilities("{T}: Add {C}{C}.\n{5}{G}{G}: Transform this creature.")))


Tap this permanent to add two colorless mana .
Pay 5 generic mana and two green mana to transform this creature.


## Carregar o dicionario e a lista de ability words

In [7]:
with open(KEYWORD_DICT_PATH, encoding="utf-8") as f:
    kw_dict = json.load(f)
with open(ABILITY_WORDS_PATH, encoding="utf-8") as f:
    ability_words = set(json.load(f))

print(f"{len(kw_dict)} termos no dicionario, {len(ability_words)} ability words")

749 termos no dicionario, 61 ability words


## Casar as keywords da carta com o dicionario

Testando contra os dados reais, o campo `keywords` do Scryfall nem sempre
bate letra por letra com as chaves do dicionario:
- **Diferenca de maiuscula/minuscula** (`"First strike"` vs `"First
  Strike"`): tratada de forma automatica via `.lower()` -- isso e' uma
  normalizacao generica correta, nao uma tabela de excecao disfarcada, e
  nao faria sentido documentar cada um dos ~200 pares manualmente.
- **Tipos de token** (`"Treasure"` vs `"Treasure Token"`): esses SIM sao
  excecoes irregulares de verdade (nomes diferentes, nao so capitalizacao),
  entao viram uma tabela explicita `TOKEN_ALIASES` -- auditavel e facil de
  estender se aparecer um token novo.

In [8]:
TOKEN_ALIASES = {
    "treasure": "Treasure Token",
    "food": "Food Token",
    "clue": "Clue Token",
    "blood": "Blood Token",
    "map": "Map Token",
    "gold": "Gold Token",
    "junk": "Junk Token",
    "powerstone": "Powerstone Token",
    "incubator": "Incubator Token",
    "role token": "Role",
}


def build_keyword_lookup(kw_dict: dict):
    lower_map = {k.lower(): k for k in kw_dict}

    def lookup(keyword):
        if keyword in kw_dict:
            return kw_dict[keyword]
        if keyword.lower() in lower_map:
            return kw_dict[lower_map[keyword.lower()]]
        if keyword.lower() in TOKEN_ALIASES:
            alias = TOKEN_ALIASES[keyword.lower()]
            if alias in kw_dict:
                return kw_dict[alias]
        # fallback: variantes parametrizadas tipo "Hexproof from" que o
        # Scryfall registra como keyword propria (nao "Hexproof" + um
        # parametro separado) -- tenta a base antes de "from"/"with".
        for sep in (" from", " with"):
            if sep in keyword.lower():
                base = keyword[: keyword.lower().index(sep)].strip()
                if base in kw_dict:
                    return kw_dict[base]
                if base.lower() in lower_map:
                    return kw_dict[lower_map[base.lower()]]
        return None

    return lookup


lookup = build_keyword_lookup(kw_dict)

## Extrair o parametro especifico da keyword na carta

Uma definicao generica de `Cycling` nao diz que ESSA carta especifica tem
`Cycling {2}`, e a definicao generica de `Protection` nao diz que essa
carta da protecao especificamente contra preto. `extract_keyword_instance`
procura o trecho da carta logo apos o nome da keyword, ate a proxima
pontuacao/parenteses de reminder text, pra capturar esse parametro quando
existe.

**Limitacao conhecida, testada e documentada em vez de escondida:**
funciona bem pra keywords com parametro escrito logo apos o nome (Cycling
{2}, Ward {4}, Protection from black, Kicker {1}{G}). NAO funciona pra
`Landwalk`, porque o Scryfall guarda o nome generico no campo `keywords`
mas o `oracle_text` da carta usa o nome especifico (`Swampwalk`,
`Islandwalk`...) -- nesse caso a funcao simplesmente nao acha nada e a
`instance` fica `None`, sem quebrar o resto do pipeline.

In [9]:
def extract_keyword_instance(keyword: str, card_text: str):
    pattern = re.compile(rf"\b{re.escape(keyword)}\b[^\n,.;(]*", re.IGNORECASE)
    m = pattern.search(card_text)
    if not m:
        return None
    span = m.group(0).strip()
    if span.lower() == keyword.lower():
        return None  # sem parametro alem do proprio nome
    return span

## Buscar as otags (etiquetas funcionais) no Scryfall

As otags sao etiquetas de **funcao** da carta (`ramp`, `removal`,
`card-advantage`...), curadas pela comunidade no Scryfall Tagger. Nao vem
no bulk data, so na busca da API: `otag:ramp` devolve todas as cartas com
a tag. Buscamos cada tag em `OTAGS` e invertemos o resultado, guardando
carta -> lista de tags.

Detalhes:
- A API exige `User-Agent` e `Accept`, e a busca e' um endpoint "pesado":
  esperamos 0,5 s entre paginas (ate 175 cartas por pagina) e, se vier
  429, esperamos e tentamos de novo.
- `unique=cards` devolve uma entrada por carta de oraculo (igual ao bulk
  "Oracle Cards"), entao o `name` casa direto com o do corpus, inclusive
  as cartas de dupla face (`"Frente // Verso"`).
- Cada tag e' salva em cache (`OTAGS_CACHE_DIR`); rodar de novo nao refaz
  as requisicoes. Tag que volta com 0 cartas provavelmente tem o nome
  errado -- o codigo avisa e nao guarda cache dela.
- **Limitacao:** a curadoria e' da comunidade, entao carta sem tag nao
  quer dizer que ela nao tenha aquela funcao (cartas novas ou obscuras
  costumam ficar sem tag).

In [10]:
import time
from collections import defaultdict
from datetime import date

import requests

SEARCH_URL = "https://api.scryfall.com/cards/search"
HEADERS = {"User-Agent": "PLN-MTG-Ilum/0.1", "Accept": "application/json"}
REQUEST_DELAY = 0.5  # segundos entre requisicoes


def scryfall_search_names(query: str, max_retries: int = 5) -> list:
    """Pagina /cards/search e devolve o nome de todas as cartas do resultado."""
    url, params, names = SEARCH_URL, {"q": query, "unique": "cards"}, []
    while url:
        for attempt in range(max_retries):
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:            # rate limit: espera e tenta de novo
                time.sleep(2 ** attempt)
                continue
            break
        if r.status_code == 404:                # busca sem nenhum resultado
            return []
        r.raise_for_status()
        data = r.json()
        names += [c["name"] for c in data["data"]]
        url = data.get("next_page") if data.get("has_more") else None
        params = None                           # next_page ja traz a query embutida
        time.sleep(REQUEST_DELAY)
    return names


os.makedirs(OTAGS_CACHE_DIR, exist_ok=True)
card_tags = defaultdict(list)

for tag in OTAGS:
    cache_path = os.path.join(OTAGS_CACHE_DIR, f"{tag}.json")
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            cached = json.load(f)
        names, origem = cached["names"], f"cache de {cached['fetched_at']}"
    else:
        names, origem = scryfall_search_names(f"otag:{tag}"), "API"
        if names:
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(
                    {"tag": tag, "query": f"otag:{tag}", "fetched_at": date.today().isoformat(), "names": names},
                    f, ensure_ascii=False,
                )
    if not names:
        print(f"ATENCAO: otag:{tag} nao retornou nenhuma carta -- confira o nome da tag no Tagger")
        continue
    for name in names:
        if tag not in card_tags[name]:
            card_tags[name].append(tag)
    print(f"otag:{tag}: {len(names)} cartas ({origem})")

card_tags = dict(card_tags)
with open(OTAGS_PATH, "w", encoding="utf-8") as f:
    json.dump(card_tags, f, ensure_ascii=False)
print(f"{len(card_tags)} cartas com pelo menos uma tag, salvas em {OTAGS_PATH}")


otag:ramp: 2315 cartas (cache de 2026-09-21)
otag:card-advantage: 6269 cartas (cache de 2026-09-21)
otag:removal: 6480 cartas (cache de 2026-09-21)
13682 cartas com pelo menos uma tag, salvas em Dados/Etapa1_card_expanded/card_otags.json


## Montar o texto de embedding (oracle + keywords + otags)

A auditoria apontou dois riscos opostos: (1) so `oracle_text` deixa de fora
o significado das keywords; (2) injetar a definicao inteira de 10 keywords
numa carta pode fazer o embedding representar mais o vocabulario generico
das regras do que a identidade especifica da carta.

Nesta rodada ficou **um unico texto de embedding**, com tres blocos (so os
que existirem pra carta):
- `Oracle Text`: o texto da carta, com os simbolos decodificados e as
  habilidades ativadas reescritas (`custo: efeito` -> `custo to efeito`)
- `Keyword Definitions`: definicao de cada keyword da carta, com o
  parametro especifico quando existe (`Cycling {2}`)
- `Functions`: as otags da carta (`ramp`, `card advantage`, `removal`),
  com o hifen trocado por espaco pra ler como palavras normais

Nome, tipo, custo de mana e cores **nao entram** no texto -- so no
registro, como metadados. Assim duas cartas com o mesmo efeito (Llanowar
Elves e Fyndhorn Elves) geram exatamente o mesmo texto de embedding.

In [11]:
def build_keyword_definitions(card: dict, card_text: str):
    """Usada so internamente pra montar o texto estruturado -- nao e' mais
    salva no registro final (o conteudo dela ja fica dentro do texto)."""
    definitions = []
    for kw in card.get("keywords") or []:
        entry = lookup(kw)
        if entry:
            definitions.append({
                "keyword": kw,
                "instance": extract_keyword_instance(kw, card_text),
                "definition": entry["definition"],
                "source": entry["source"],
                "category": entry["category"],
            })
    return definitions


def build_embedding_text(card_text: str, keyword_defs: list, tags: list) -> str:
    """Oracle + significado das keywords + otags. Nada de nome/tipo/custo/cor."""
    oracle = expand_symbols(restructure_activated_abilities(card_text))

    sections = []
    if oracle:
        sections.append(f"Oracle Text:\n{oracle}")
    if keyword_defs:
        kw_lines = "\n".join(
            f"- {d['keyword']}" + (f" ({d['instance']})" if d["instance"] else "") + f": {d['definition']}"
            for d in keyword_defs
        )
        sections.append(f"Keyword Definitions:\n{kw_lines}")
    if tags:
        sections.append("Functions: " + ", ".join(t.replace("-", " ") for t in tags))
    return "\n".join(sections)


## Processar todas as cartas (um unico passe pelo arquivo)

Cartas com `layout` em `EXCLUDED_LAYOUTS` ou `set_type` em
`EXCLUDED_SET_TYPES` sao puladas antes de qualquer processamento -- nao
entram no corpus de busca (Un-sets removidos por decisao revisada).

O registro final nao guarda `id`/`oracle_id` nem `keyword_definitions` --
o nome vira a referencia da carta, e as definicoes ja estao dentro do
texto estruturado. Os misses de keyword continuam classificados em A
(ability word)/B (gap real)/C (token especial), igual antes.

Cada registro ganha tambem `otags` (lista, pode ser vazia) e o campo unico
`embedding_text`. Cartas sem texto de efeito (ex.: criaturas vanilla)
recebem o texto `vanilla` em `oracle_text` (e, em cartas de duas faces, em
cada face sem texto), entao o `embedding_text` nunca fica vazio e elas entram
normalmente na etapa de embeddings. O resumo final conta quantas sao.

In [12]:
def classify_miss(keyword: str) -> str:
    # normaliza apostrofo reto vs curvo -- Scryfall usa reto ('), a CR usa
    # curvo (’) no texto oficial, e sao caracteres diferentes na comparacao
    kw_lower = keyword.lower().replace("’", "'")
    ability_words_norm = {w.replace("’", "'") for w in ability_words}
    if kw_lower in ability_words_norm or any(aw.startswith(kw_lower + " ") for aw in ability_words_norm):
        return "A_ability_word"
    if "token" in kw_lower:
        return "C_token_especial"
    return "B_gap_real"


missing_counter = {}
missing_by_category = {"A_ability_word": 0, "B_gap_real": 0, "C_token_especial": 0}
total_kw_mentions = 0
n_cards = 0
n_excluded = 0
n_tagged = 0
n_vanilla = 0
tag_counts = {tag: 0 for tag in OTAGS}
corpus_names = set()

with open(CARDS_PATH, encoding="utf-8") as fin, open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        card = json.loads(line)
        if card.get("layout") in EXCLUDED_LAYOUTS or card.get("set_type") in EXCLUDED_SET_TYPES:
            n_excluded += 1
            continue

        card_text = get_card_text(card)
        keywords = card.get("keywords") or []
        total_kw_mentions += len(keywords)

        keyword_defs = build_keyword_definitions(card, card_text)
        matched_names = {d["keyword"] for d in keyword_defs}
        for kw in keywords:
            if kw not in matched_names:
                missing_counter[kw] = missing_counter.get(kw, 0) + 1
                missing_by_category[classify_miss(kw)] += 1

        tags = card_tags.get(card.get("name"), [])
        embedding_text = build_embedding_text(card_text, keyword_defs, tags)

        record = {
            "name": card.get("name"),
            "type_line": card.get("type_line"),
            "mana_cost": card.get("mana_cost"),
            "colors": card.get("colors"),
            "power": card.get("power"),
            "toughness": card.get("toughness"),
            "loyalty": card.get("loyalty"),
            "defense": card.get("defense"),
            "produced_mana": card.get("produced_mana"),
            "oracle_text": card_text,
            "faces": get_card_faces(card),
            "keywords": keywords,
            "otags": tags,
            "embedding_text": embedding_text,
        }
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")
        n_cards += 1
        corpus_names.add(card.get("name"))
        if tags:
            n_tagged += 1
            for t in tags:
                tag_counts[t] = tag_counts.get(t, 0) + 1
        if set(card_text.replace("//", " ").split()) == {"vanilla"}:
            n_vanilla += 1

n_missing = sum(missing_counter.values())
print(f"{n_excluded} cartas excluidas (layout/set_type fora do escopo de baralho)")
print(f"{n_cards} cartas processadas, salvas em {OUTPUT_PATH}")
print(f"{total_kw_mentions - n_missing}/{total_kw_mentions} mencoes de keyword expandidas ({(total_kw_mentions - n_missing)/total_kw_mentions*100:.1f}%)")
print()
print("classificacao dos misses:")
for cat, n in missing_by_category.items():
    print(f"  {cat}: {n} mencoes")
print()
print(f"{n_tagged} cartas com pelo menos uma otag ({n_tagged/n_cards*100:.1f}%) -- por tag: {tag_counts}")
not_in_corpus = set(card_tags) - corpus_names
print(f"{len(not_in_corpus)} cartas com otag nao estao no corpus (layout/set_type excluidos ou nome divergente); exemplos: {sorted(not_in_corpus)[:5]}")
print(f"{n_vanilla} cartas sem texto de efeito (ex.: criaturas vanilla): recebem o texto 'vanilla' e entram normalmente nos embeddings")


5479 cartas excluidas (layout/set_type fora do escopo de baralho)
33147 cartas processadas, salvas em Dados/Etapa1_card_expanded/card_documents.jsonl
21320/23623 mencoes de keyword expandidas (90.3%)

classificacao dos misses:
  A_ability_word: 1239 mencoes
  B_gap_real: 1064 mencoes
  C_token_especial: 0 mencoes

13336 cartas com pelo menos uma otag (40.2%) -- por tag: {'ramp': 2255, 'card-advantage': 6118, 'removal': 6300}
346 cartas com otag nao estao no corpus (layout/set_type excluidos ou nome divergente); exemplos: ['"Name Sticker" Goblin', '"Rumors of My Death . . ."', 'AWOL', 'Acornelia, Fashionable Filcher', 'Aerid Konstrari']


## Conferindo alguns exemplos

In [13]:
with open(OUTPUT_PATH, encoding="utf-8") as f:
    cards = [json.loads(l) for l in f]
by_name = {c["name"]: c for c in cards}

print("=== confirmando que a Llanowar Elves token sumiu (era o problema original) ===")
llanowar_matches = [c for c in cards if c["name"] == "Llanowar Elves"]
print(f"{len(llanowar_matches)} registro(s) chamado(s) Llanowar Elves:", [c["type_line"] for c in llanowar_matches])
print()

print("=== carta de dupla face, faces preservadas ===")
dfc = by_name["Ulvenwald Captive // Ulvenwald Abomination"]
for face in dfc["faces"]:
    print(" -", face["name"], "|", face["oracle_text"][:60])
print()

print("=== texto de embedding (oracle + keywords + otags) ===")
for name in ["Llanowar Elves", "Swords to Plowshares", "Serra Angel"]:
    card = by_name.get(name)
    if card is None:
        print(f"--- {name}: nao encontrada no corpus ---")
        continue
    print(f"--- {name} | otags: {card['otags']} ---")
    print(card["embedding_text"])
    print()

a, b = by_name.get("Llanowar Elves"), by_name.get("Fyndhorn Elves")
if a and b:
    print("Llanowar Elves e Fyndhorn Elves com texto de embedding identico?", a["embedding_text"] == b["embedding_text"])
print()
print("campos no registro:", list(cards[0].keys()))


=== confirmando que a Llanowar Elves token sumiu (era o problema original) ===
1 registro(s) chamado(s) Llanowar Elves: ['Creature — Elf Druid']

=== carta de dupla face, faces preservadas ===
 - Ulvenwald Captive | Defender
{T}: Add {G}.
{5}{G}{G}: Transform this creature.
 - Ulvenwald Abomination | {T}: Add {C}{C}.

=== texto de embedding (oracle + keywords + otags) ===
--- Llanowar Elves | otags: ['ramp'] ---
Oracle Text:
Tap this permanent to add green mana .
Functions: ramp

--- Swords to Plowshares | otags: ['removal'] ---
Oracle Text:
Exile target creature. Its controller gains life equal to its power.
Functions: removal

--- Serra Angel | otags: [] ---
Oracle Text:
Flying
Vigilance (Attacking doesn't cause this creature to tap.)
Keyword Definitions:
- Flying: Flying is an evasion ability. A creature with flying cant be blocked except by creatures with flying and/or reach. A creature with flying can block a creature with or without flying. Multiple instances of flying on the sam